# Track C (Weather and Climate): starter notebook - simulated extreme rainfall

Predict the probability of heavy rain **tomorrow** (`heavy_rain_tomorrow`). Primary metric: **average precision**. The test set is **later in time** (2021-2024) and includes **unseen stations**, so validation should respect time order.

*IML Midterm Project, CS-3410 / MAT-3211-1, Ashoka University. Data are simulated.*

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

SEED = 0
SLUG = "iml2026-track-c-simulated-extreme-rain"
# Works on Kaggle (dataset attached), locally (./data/<folder>) or via IML_DATA_DIR.
candidates = [os.environ.get("IML_DATA_DIR", ""), f"/kaggle/input/{SLUG}", "data/track_c_weather_extreme_rainfall", "data"]
DATA = next(Path(c) for c in candidates if c and (Path(c) / "train.csv").exists())
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
print(DATA, train.shape, test.shape)
train.head()

## 1. Remove what you will not have at prediction time
`rain_tomorrow_mm` is tomorrow's rain, the quantity that defines the label. It is absent from `test.csv`. Drop it.

In [ ]:
ID, TARGET = "record_id", "heavy_rain_tomorrow"
train = train.drop(columns=["rain_tomorrow_mm"])
train["date"] = pd.to_datetime(train["date"])
print(train[TARGET].mean().round(4), train.groupby(train.date.dt.year)[TARGET].mean().round(3).to_dict())

## 2. A validation split that respects time
Train on 2010-2017 and validate on 2018-2020. Never let the model train on days that come after the days it is validated on.

In [ ]:
from sklearn.metrics import average_precision_score
fit_rows = train.date < "2018-01-01"
val_rows = ~fit_rows
y = train[TARGET]

def evaluate(model, X):
    model.fit(X[fit_rows], y[fit_rows])
    return round(average_precision_score(y[val_rows], model.predict_proba(X[val_rows])[:, 1]), 3)

## 3. Trivial baseline
A constant prediction scores the prevalence of the validation period.

In [ ]:
FEATURES = ["mslp_hpa", "pressure_change_24h_hpa", "rh_1500_pct", "precipitable_water_mm"]
print("constant:", evaluate(DummyClassifier(strategy="prior"), train[FEATURES]))

## 4. A deliberately small first model
Four features, imputed and scaled inside the pipeline.

In [ ]:
model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000))
print("small logistic:", evaluate(model, train[FEATURES]))

## 5. Fit on all training data and write the submission

In [ ]:
model.fit(train[FEATURES], y)
submission = pd.DataFrame({ID: test[ID], TARGET: model.predict_proba(test[FEATURES])[:, 1]})

In [ ]:
# Check the file against the required format before uploading it.
assert list(submission.columns) == list(sample.columns), "columns must match sample_submission.csv"
assert len(submission) == len(sample) and set(submission[ID]) == set(sample[ID]), "every test id exactly once"
assert submission[TARGET].notna().all()
submission.to_csv("submission.csv", index=False)
print("wrote submission.csv", submission.shape)
submission.head()

## Where to go from here
This notebook gives you a sound *protocol*, not a good model. Your project adds a research question, at least three baselines, at least two controlled experiments, error analysis and limitations (see the project guide). Keep every new idea inside the same validation scheme. The data are **simulated**; say so in your report.